# MNIST Digits: Topological Features

Topology can distinguish **how many holes** a digit has, but not which digit it is.

- β₁ = 0: digits 1, 2, 3, 5, 7 (no enclosed regions)
- β₁ = 1: digits 0, 4, 6, 9 (one loop)
- β₁ = 2: digit 8 (two loops)

Within each group, topology is blind — a 0 and a 4 and a 6 are all topologically the same. What TDA gives you is a **robust feature** (invariant to stretching and noise) that complements standard ML approaches.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/avacadobanana352/tda-witness/blob/main/examples/mnist_topology.ipynb)

In [ ]:
!pip install -q --no-cache-dir "tda-witness[all] @ git+https://github.com/avacadobanana352/tda-witness.git"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tda import analyze
from tda.visualization import plot_complex, save_html

## Load MNIST

Download the test set (10k images) and convert each 28x28 grayscale image to a 2D point cloud by taking the (x, y) coordinates of bright pixels.

In [ ]:
import urllib.request, gzip, struct, os, ssl

# Download MNIST test set
url = "https://ossci-datasets.s3.amazonaws.com/mnist/"
cache = "/tmp/mnist"
os.makedirs(cache, exist_ok=True)

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

for fname in ["t10k-images-idx3-ubyte.gz", "t10k-labels-idx1-ubyte.gz"]:
    path = os.path.join(cache, fname)
    if not os.path.exists(path):
        print(f"Downloading {fname}...")
        req = urllib.request.Request(url + fname)
        with urllib.request.urlopen(req, context=ctx) as resp, open(path, "wb") as f:
            f.write(resp.read())

with gzip.open(os.path.join(cache, "t10k-images-idx3-ubyte.gz"), "rb") as f:
    _, n, rows, cols = struct.unpack(">IIII", f.read(16))
    images = np.frombuffer(f.read(), dtype=np.uint8).reshape(n, rows, cols)

with gzip.open(os.path.join(cache, "t10k-labels-idx1-ubyte.gz"), "rb") as f:
    _, n = struct.unpack(">II", f.read(8))
    labels = np.frombuffer(f.read(), dtype=np.uint8)

print(f"Loaded {len(images)} images ({rows}x{cols})")

In [ ]:
def image_to_point_cloud(img, threshold=128):
    """Convert a 28x28 grayscale image to a 2D point cloud."""
    ys, xs = np.where(img > threshold)
    ys = (rows - 1 - ys)  # flip y so it's right-side up
    return np.column_stack([xs, ys]).astype(float)

## The topological signature

Digits fall into exactly three topological classes based on β₁ (number of loops):

| β₁ | Digits | Why |
|:--:|--------|-----|
| 0 | 1, 2, 3, 5, 7 | Open strokes, no enclosed region |
| 1 | 0, 4, 6, 9 | One enclosed region |
| 2 | 8 | Two enclosed regions |

**Limitation:** topology cannot distinguish within each group. A 0, 4, 6, and 9 are identical by this measure. What we *can* reliably detect is the number of holes — which is a useful feature, just not a complete classifier.

## Visualize a few digits as point clouds

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for digit in range(10):
    idx = np.where(labels == digit)[0][0]
    pc = image_to_point_cloud(images[idx])
    ax = axes[digit // 5, digit % 5]
    ax.scatter(pc[:, 0], pc[:, 1], s=8, c="royalblue")
    ax.set_title(f"Digit {digit} ({len(pc)} pts)", fontsize=12)
    ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle("MNIST digits as 2D point clouds", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Compute Betti numbers

Run `analyze()` on each digit's point cloud. The key parameter is the threshold `R` — set it just large enough to connect neighboring pixels (~1.5 pixel units) but small enough to preserve holes.

In [ ]:
R = 1.5
expected_b1 = {0:1, 1:0, 2:0, 3:0, 4:1, 5:0, 6:1, 7:0, 8:2, 9:1}

print(f"Threshold R={R}, using up to 40 landmarks per digit\n")
print("digit | n_pts | B0  B1  | expected B1 | match")
print("------+-------+---------+-------------+------")

results = {}
for digit in range(10):
    idx = np.where(labels == digit)[0][0]
    pc = image_to_point_cloud(images[idx])
    n_lm = min(40, len(pc))
    result = analyze(pc, threshold=R, simplex_dim=2, n_landmarks=n_lm, normalize=False, seed=42)
    results[digit] = (pc, result)

    betti = result["betti"]
    while len(betti) < 2:
        betti.append(0)
    match = "Y" if betti[1] == expected_b1[digit] else ""
    print(f"  {digit}   | {len(pc):>4}  | {betti[0]:>2} {betti[1]:>2}  |      {expected_b1[digit]}      |  {match}")

## Visualize the simplicial complex

See the edges and triangles that the witness complex builds on each digit. The holes in the complex correspond to $\beta_1$.

In [ ]:
# Pick a few interesting digits to visualize
for digit in [0, 1, 8]:
    pc, result = results[digit]
    betti = result["betti"]
    while len(betti) < 2:
        betti.append(0)
    fig = plot_complex(result["data"], result["landmarks"], result["graph"], result["complex"],
                       title=f"Digit {digit}: B0={betti[0]}, B1={betti[1]}")
    fig.show()

## How reliably is β₁ computed?

For each digit we know the expected β₁ (from the table above). This checks how consistently `tda-witness` recovers the correct loop count — it measures the reliability of the computation, **not** classification accuracy (since 0, 4, 6, 9 would all be indistinguishable even with perfect computation).

In [ ]:
N_SAMPLES = 20  # per digit

print(f"Testing {N_SAMPLES} samples per digit...\n")

total_correct = 0
total = 0
digit_acc = {}

for digit in range(10):
    idxs = np.where(labels == digit)[0][:N_SAMPLES]
    correct = 0
    for i in idxs:
        pc = image_to_point_cloud(images[i])
        n_lm = min(40, len(pc))
        res = analyze(pc, threshold=R, simplex_dim=2, n_landmarks=n_lm, normalize=False, seed=42)
        betti = res["betti"]
        while len(betti) < 2:
            betti.append(0)
        if betti[1] == expected_b1[digit]:
            correct += 1
    digit_acc[digit] = correct / len(idxs)
    total_correct += correct
    total += len(idxs)
    print(f"  Digit {digit}: {correct}/{len(idxs)} ({digit_acc[digit]:.0%})")

print(f"\nOverall: {total_correct}/{total} = {total_correct/total:.0%}")

In [ ]:
# Plot accuracy by digit
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#2ca02c" if v >= 0.8 else "#ff7f0e" if v >= 0.5 else "#d62728" for v in digit_acc.values()]
ax.bar(digit_acc.keys(), digit_acc.values(), color=colors, edgecolor="white")
ax.set_xlabel("Digit", fontsize=12)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title(f"B1 classification accuracy by digit (R={R}, {N_SAMPLES} samples each)", fontsize=13)
ax.set_xticks(range(10))
ax.set_ylim(0, 1.1)
ax.axhline(y=total_correct/total, color="gray", linestyle="--", label=f"Overall: {total_correct/total:.0%}")
ax.legend()
plt.tight_layout()
plt.show()

## What topology can and can't do here

**Can:** group digits into three topological classes (0 holes / 1 hole / 2 holes). This works with no training, no gradients, no GPU — purely from geometry.

**Can't:** distinguish within a class. A 0 and a 4 and a 6 are topologically identical (all have one loop), so β₁ alone cannot tell them apart.

**Why it's still useful:** β₁ is invariant to stretching, rotation, and small noise — unlike pixel-level features. In practice, TDA features like β₁ and persistence diagrams are used *alongside* ML models, not instead of them. They add robustness that geometric methods can't easily learn.